# Choose exactly which rows to score

Learn how to supply your own prediction population, specific customers at specific anchor times, instead of Kumo's default.
This Kumo Engine module trains two small models on synthetic retail data (binary and merchant ranking), scores a population you write and shows how many rows come back and which errors to expect.

Use a Linux notebook prepared with the [Kumo Engine setup](https://docs.nvidia.com/sdgm/kumo-engine/get-started-with-kumo-engine/installation/linux-machine-deployment). A GPU is recommended for the FAST training preset; set `SMOKE_TEST = True` below for a quick check on a CPU-only machine.
Expect under a minute on a single L4 GPU after installation.

See [Making predictions](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/making-predictions#select-the-population) for the supported ways to select a population.

## Install

Install Kumo Engine in this notebook kernel.

In [1]:
%pip install kumo-engine

## Choose a workspace

Set `KUMO_LOCAL_SSD` to a local SSD mount for faster scratch storage; otherwise the system temporary directory is used.
This cell creates a new directory on each run, with separate source and scratch directories, and does not replace existing data.
The last lines print the installed Kumo Engine version and the workspace.

In [2]:
import importlib.metadata
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from kumoapi.jobs import MetadataField
from kumoapi.model_plan import RunMode
from kumoapi.typing import Stype

import kumo.engine as kumo

workspace_root = Path(os.environ.get("KUMO_LOCAL_SSD", tempfile.gettempdir()))
workspace = Path(tempfile.mkdtemp(prefix="kumo-custom-prediction-", dir=workspace_root))
source_dir = workspace / "source"
source_dir.mkdir()
kumo.configure(scratch_dir=workspace / "scratch")
print("Kumo Engine:", importlib.metadata.version("kumo-engine"))
print("Workspace:", workspace)

Kumo Engine: 1.0.1
Workspace: /tmp/kumo-custom-prediction-rewtib37


## Set the training budget

Models train with the [FAST preset](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/model-plan-guide).
Set `SMOKE_TEST = True` to cap training at two short epochs for a quick check on a CPU-only machine; its scores carry no meaning.

In [3]:
SMOKE_TEST = False

## Generate synthetic example data

Run the next cell to create synthetic customer, merchant and transaction tables: 500 customers, 12 merchants and 1,500 transactions over six months.
You can skim the generation code; the Kumo workflow starts below.

In [4]:
rng = np.random.default_rng(0)
pd.DataFrame({"CUSTOMER_ID": range(500), "AGE": rng.integers(20, 80, 500)}
             ).to_parquet(source_dir / "customers.parquet", index=False)
pd.DataFrame({"MERCHANT_ID": range(12), "CATEGORY": [f"category_{i % 4}" for i in range(12)]}
             ).to_parquet(source_dir / "merchants.parquet", index=False)
pd.DataFrame({
    "TX_ID": range(1500),
    "CUSTOMER_ID": rng.integers(0, 500, 1500),
    "MERCHANT_ID": rng.integers(0, 12, 1500),
    "TS": pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 180, 1500), unit="D"),
    "AMOUNT": rng.gamma(2.0, 40.0, 1500).round(2),
}).to_parquet(source_dir / "transactions.parquet", index=False)

## Build the graph

Register the three tables and the two relationships from each transaction to its customer and merchant.
See [Graphs](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/graphs) and [Tables](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/set-up-a-graph/tables).

In [5]:
connector = kumo.FileConnector(root_dir=str(source_dir))
customers = kumo.Table.from_source_table(
    connector["customers.parquet"], primary_key="CUSTOMER_ID"
).infer_metadata()
merchants = kumo.Table.from_source_table(
    connector["merchants.parquet"], primary_key="MERCHANT_ID"
).infer_metadata()
transactions = kumo.Table.from_source_table(
    connector["transactions.parquet"], primary_key="TX_ID", time_column="TS"
).infer_metadata().with_column("CUSTOMER_ID", stype=Stype.ID).with_column("MERCHANT_ID", stype=Stype.ID)
graph = kumo.Graph(
    tables={"customers": customers, "merchants": merchants, "transactions": transactions},
    edges=[
        kumo.Edge(src_table="transactions", fkey="CUSTOMER_ID", dst_table="customers"),
        kumo.Edge(src_table="transactions", fkey="MERCHANT_ID", dst_table="merchants"),
    ],
)
graph.validate()
snapshot = graph.snapshot()

## Define the queries

The binary query asks whether a customer transacts in the next 30 days. The ranking query asks which three merchants they buy from.
Both are temporal, so each prediction row is a customer at an anchor time. See [Predictive Queries](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/predictive-queries).
The task types should be `binary_classification` and `temporal_link_prediction`.

In [6]:
binary_query = kumo.PredictiveQuery(
    graph, "PREDICT COUNT(transactions.*, 0, 30, days) > 0 FOR EACH customers.CUSTOMER_ID"
)
ranking_query = kumo.PredictiveQuery(
    graph,
    "PREDICT LIST_DISTINCT(transactions.MERCHANT_ID, 0, 30, days) RANK TOP 3 FOR EACH customers.CUSTOMER_ID",
)
print(binary_query.get_task_type(), "|", ranking_query.get_task_type())

binary_classification | temporal_link_prediction


## Train both models

The FAST plan uses one experiment without refitting, and explanations are disabled, to keep this module on prediction tables rather than tuning.
Keep each trainer: its `training_table_generation_plan` is passed to prediction tables built later so they match the fitted model. See [Training](https://docs.nvidia.com/sdgm/kumo-engine/user-code-guide/train-a-model/training).

In [7]:
def fast_plan(query, snapshot, **suggest_options):
    model_plan = query.suggest_model_plan(snapshot, run_mode=RunMode.FAST, **suggest_options)
    model_plan.training_job.num_experiments = 1
    model_plan.training_job.refit_trainval = False
    model_plan.training_job.refit_full = False
    model_plan.training_job.disable_explain = True
    if SMOKE_TEST:
        model_plan.optimization.max_epochs = 2
        model_plan.optimization.max_steps_per_epoch = 20
    return model_plan


def train(query):
    trainer = kumo.Trainer(model_plan=fast_plan(query, snapshot))
    trainer.fit(query.generate_training_table(snapshot), snapshot=snapshot)
    return trainer


binary_trainer = train(binary_query)
ranking_trainer = train(ranking_query)

## Supply a population

A temporal prediction table has an `ENTITY1` column and a timezone-naive `TIMESTAMP` column holding each row's anchor time. It must not contain `TARGET1`.
Write five customers at two anchor times each, as a file the connector can read.

In [8]:
population = pd.DataFrame({
    "ENTITY1": np.repeat(range(5), 2),
    "TIMESTAMP": pd.to_datetime(["2024-06-01", "2024-06-08"] * 5),
})
population.to_parquet(source_dir / "population.parquet", index=False)

Build the prediction table from that file and score it. Expect ten rows in and ten rows out, one prediction per input row.

In [9]:
prediction_table = kumo.PredictionTable.from_source_table(
    connector["population.parquet"],
    pq=binary_query,
    snapshot=snapshot,
    plan=binary_trainer.training_table_generation_plan,
)
predictions = binary_trainer.predict(
    prediction_table, snapshot=snapshot, binary_classification_threshold=0.5
).frame
print(f"{prediction_table.row_count} rows in, {len(predictions)} rows out")
predictions.head(4)

10 rows in, 10 rows out


,ENTITY,TARGET_PRED,False_PROB,True_PROB
0,0,False,0.643712,0.356288
1,1,False,0.626230,0.373770
2,2,False,0.677124,0.322876
3,3,False,0.669967,0.330033


Duplicate rows are not removed, and the default output does not say which anchor a row belongs to.
Request it with `extra_fields`, which writes the predictions through a connector. Expect each customer to appear once per anchor time.

In [10]:
output_config = kumo.PredictionOutputConfig(
    connector=kumo.FileConnector(root_dir=str(workspace / "outputs")),
    output_paths={"predictions": "binary/"},
    extra_fields=[MetadataField.ANCHOR_TIMESTAMP],
)
binary_trainer.predict(
    prediction_table,
    snapshot=snapshot,
    output_config=output_config,
    binary_classification_threshold=0.5,
)
parts = sorted((workspace / "outputs" / "binary").glob("*.parquet"))
pd.concat(map(pd.read_parquet, parts)).sort_values(["ENTITY", "ANCHOR_TIMESTAMP"]).head(4)

,ENTITY,ANCHOR_TIMESTAMP,TARGET_PRED,False_PROB,True_PROB
0,0,2024-06-01,False,0.643712,0.356288
5,0,2024-06-08,False,0.634778,0.365222
1,1,2024-06-01,False,0.626230,0.373770
6,1,2024-06-08,False,0.633224,0.366776


## Output rows can exceed input rows

Ranking and multiclass tasks return one row per input row and returned class. Score six customers with `num_classes_to_return=3` and expect 18 rows.

In [11]:
pd.DataFrame({"ENTITY1": range(6), "TIMESTAMP": pd.Timestamp("2024-06-01")}
             ).to_parquet(source_dir / "ranking_population.parquet", index=False)
ranking_table = kumo.PredictionTable.from_source_table(
    connector["ranking_population.parquet"],
    pq=ranking_query,
    snapshot=snapshot,
    plan=ranking_trainer.training_table_generation_plan,
)
ranked = ranking_trainer.predict(ranking_table, snapshot=snapshot, num_classes_to_return=3).frame
print(f"{ranking_table.row_count} rows in, {len(ranked)} rows out")
ranked.head(3)

6 rows in, 18 rows out


,ENTITY,CLASS,SCORE
0,0,8,0.687970
1,0,6,0.245099
2,0,11,0.038695


## Errors to expect

A `TARGET1` column and a missing `TIMESTAMP` column are both rejected, with a message naming the column.

In [12]:
for name, frame in {
    "with_target.parquet": population.assign(TARGET1=1),
    "without_timestamp.parquet": population[["ENTITY1"]],
}.items():
    frame.to_parquet(source_dir / name, index=False)
    try:
        kumo.PredictionTable.from_source_table(connector[name], pq=binary_query, snapshot=snapshot)
    except ValueError as error:
        print(error, "\n")

Target column in prediction table. A custom prediction table selects rows to score and must not contain target columns; remove ['TARGET1']. 

Required column not found. Source table is missing TIMESTAMP, which is the name DiskGraph reads that column under and the only name it reads it under. It carries ['ENTITY1']. 



## Next step

Write the population from your own customer list and the times you need scores for, then pass the trained model's `training_table_generation_plan` so the table matches the fitted model.
To score new customers after a data refresh, see [incremental scoring](local_kumo_engine_incremental_scoring.ipynb); to train on your own labels, see [temporal custom training table](local_kumo_engine_temporal_custom_training_table.ipynb).

## Cleanup

Remove this notebook's workspace, which holds only the files it created. Skip this cell to keep the generated tables and predictions.

In [13]:
import shutil

shutil.rmtree(workspace)